# 06 - ML Architecture & Rigor Challenges

This notebook focuses on the software engineering side of ML, including validation and testing.

## 1. Data Validation with Pydantic

**Task:** Create a validation schema for model input features to prevent 'garbage in, garbage out' in production.

In [ ]:
from pydantic import BaseModel, Field, validator
from typing import List

class ModelInput(BaseModel):
    user_id: int
    recent_purchases: List[float] = Field(..., min_items=1)
    age: int = Field(..., gt=0, lt=120)
    country_code: str = Field(..., min_length=2, max_length=2)

    @validator('country_code')
    def country_must_be_uppercase(cls, v):
        if v != v.upper():
            raise ValueError('country_code must be uppercase')
        return v

# Test valid input
try:
    valid_input = ModelInput(user_id=123, recent_purchases=[10.5, 20.0], age=25, country_code='US')
    print("Valid Input Accepted")
except Exception as e:
    print(f"Validation Error: {e}")

# Test invalid input
try:
    invalid_input = ModelInput(user_id=123, recent_purchases=[], age=-5, country_code='usa')
except Exception as e:
    print(f"Caught Expected Error: {e}")

## 2. Unit Testing a Custom PyTorch Layer

**Task:** Write a test to ensure a custom layer (e.g., a simple Residual Block) maintains output shape and is differentiable.

In [ ]:
import torch
import torch.nn as nn
import unittest

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        return x + self.relu(self.conv(x))

class TestMLComponents(unittest.TestCase):
    def test_residual_shape(self):
        channels = 16
        model = ResidualBlock(channels)
        x = torch.randn(1, channels, 32, 32)
        output = model(x)
        self.assertEqual(output.shape, x.shape)
        
    def test_differentiability(self):
        channels = 16
        model = ResidualBlock(channels)
        x = torch.randn(1, channels, 32, 32, requires_grad=True)
        output = model(x)
        loss = output.sum()
        loss.backward()
        self.assertIsNotNone(x.grad)

# Run tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestMLComponents)
unittest.TextTestRunner().run(suite)

## 3. Conceptual: Model Serving with FastAPI

**Task:** Simple wrapper for model inference with error handling.

In [ ]:
from fastapi import FastAPI, HTTPException
import logging

app = FastAPI()
logger = logging.getLogger("ml_service")

def dummy_model_predict(data):
    # Simulate inference
    if data['val'] < 0: raise ValueError("Negative value")
    return {"prediction": 1.0}

@app.post("/predict")
async def predict(input_data: dict):
    try:
        result = dummy_model_predict(input_data)
        return result
    except ValueError as ve:
        logger.error(f"Inference error: {ve}")
        raise HTTPException(status_code=400, detail=str(ve))
    except Exception as e:
        logger.critical(f"System error: {e}")
        raise HTTPException(status_code=500, detail="Internal Server Error")

print("FastAPI inference skeleton defined.")